In [25]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import csv
import numpy as np
import copy

carico il file dei commenti che ho estrapolato tramite il mio sondaggio per cercare di analizzare il tutto e capire che cosa ho ottenuto

## Controllo validità dati iniziali

In [26]:
df = pd.read_csv(
    "./responses_27mag_FULL.csv",
    sep=";",
    quoting=csv.QUOTE_MINIMAL,
    encoding="utf-8",
    engine="python"
)

Mi sono accorta che nel deploy ho messo poi le label sbagliate, quindi le vado a sistemare tutte nel file nuovo df

In [27]:
df['age_group'].replace('18-25', '18-29', inplace=True)
df['age_group'].replace('26-35', '30-45', inplace=True)
df['age_group'].replace('36-45', '46-60', inplace=True)
df['age_group'].replace('46', '61+', inplace=True)

In [28]:
age_group_counts = df['age_group'].value_counts()
gender_counts = df['gender'].value_counts()

age_group_counts, gender_counts

(18-29    106
 30-45     81
 46-60     22
 61+        2
 Name: age_group, dtype: int64,
 Female    115
 Male       96
 Name: gender, dtype: int64)

## PRE-PREPROCESSING

In [29]:
# Mappiamo i livelli di tossicità a numeri
tossicita_map = {
    "Non toxic": 0,
    "Non racist": 0,
    "Non harassing": 0,
    "Non vulgar": 0,
    "Non violent": 0,

    "A little toxic": 1,
    "A little racist": 1,
    "A little harassing": 1,
    "A little vulgar": 1,
    "A little violent": 1,

    "Toxic": 2,
    "Racist": 2,
    "Harassing": 2,
    "Vulgar": 2,
    "Violent": 2,

    "Extremely toxic": 3,
    "Extremely racist": 3,
    "Extremely harassing": 3,
    "Extremely vulgar": 3 ,
    "Extremely violent": 3
}

In [30]:
# Mappiamo la tipologia di tossicità
# 0: Toxicity
# 1: Racism
# 2: Harassment
# 3: Vulgarity      
# 4: Violence

type_toss_map = {
    "Non toxic": 0, 
    "A little toxic": 0,
    "Toxic": 0,
    "Extremely toxic": 0,

    "Non racist": 1,
    "A little racist": 1,
    "Racist": 1,
    "Extremely racist": 1,

    "Non harassing": 2,
    "A little harassing": 2,
    "Harassing": 2,
    "Extremely harassing": 2,

    "Non vulgar": 3,
    "A little vulgar": 3,
    "Vulgar": 3,
    "Extremely vulgar": 3,

    "Non violent": 4,
    "A little violent": 4,
    "Violent": 4,
    "Extremely violent": 4
}

In [31]:
# Codifica gender
gender_map = {
    'Male': 0, 
    'Female': 1}
df['gender'] = df['gender'].map(gender_map)

In [32]:
# Codifica age_group
age_map = {
    '18-29': 1,
    '30-45': 2, 
    '46-60': 3, 
    '61+': 4}
df['age_group'] = df['age_group'].map(age_map)

In [33]:
 # Escludi le prime 3 colonne: 'id', 'age_group', 'gender'
domande_colonne = [col for col in df.columns if 'q' in col and 'text' in col]
risposte_colonne = [col for col in df.columns if 'q' in col and 'response' in col]

In [34]:
data = []
for i, row in df.iterrows():
    for j in range(len(domande_colonne)):  # Per ogni domanda
        domanda = row[domande_colonne[j]]
        risposta = row[risposte_colonne[j]]
        tossicita = tossicita_map.get(risposta, -1)  # Mappa la risposta alla tossicità
        data.append({
            'commento': domanda,
            'valutazione': risposta,
            'tossicita': tossicita,
            'tipo_tossicita': type_toss_map.get(risposta, -1),  # Mappa la risposta al tipo di tossicità
            'age_group': row['age_group'],
            'gender': row['gender'],
            'utente_id': row['id'] if 'id' in row else i  # se esiste un ID, altrimenti usa i
        })

df = pd.DataFrame(data)

Va bene perchè sono 25 commenti da valutare, quindi 211*25 = 5275 commenti in tutto (anche ripetuti)


### controlli occorrenze

In [35]:
# Conta le occorrenze di ciascuna valutazione 
conta_risposte = df['valutazione'].value_counts()
conta_risposte

Non toxic              918
Extremely violent      647
Racist                 472
A little harassing     469
A little vulgar        389
Harassing              376
A little racist        366
Vulgar                 354
Violent                286
Non vulgar             241
Non harassing          153
Extremely racist       109
A little violent       109
Non racist              95
A little toxic          85
Extremely vulgar        80
Toxic                   47
Extremely harassing     45
Non violent             25
Extremely toxic          9
Name: valutazione, dtype: int64

In [36]:
# Conta le occorrenze di ciascun commento
conta_domanda = df['commento'].value_counts()
conta_domanda

Death to capitalism                                                                                                                                                                                                 33
Traps are gay.                                                                                                                                                                                                      30
Nazi punk fuck off                                                                                                                                                                                                  15
Kill them. Kill them now                                                                                                                                                                                            14
Coffee oatmeal with low cal sweetner is the shit                                                                                            

Possiamo quindi dire che sono stati annotati in maniera differente 979 diversi commenti (commenti unici)

In [37]:
# seleziono i commenti che compiaiono almeno 3 volte
domande_min3 = conta_domanda[conta_domanda >= 3]
domande_min3

Death to capitalism                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             33
Traps are gay.                                                                                                                                                                                                                                                                                                                                                                                                       

Su un totale di 979 commenti, ci sono 890 commenti che sono stati valutati almeno 3 volte --> possiamo fare la media della loro valutazione per estrarne la tossicità

In [38]:
# adesso mi seleziono i commenti che sono comparsi un numero minore di 3 volte
commenti_non_3 = conta_domanda[conta_domanda < 3].index.tolist()
print(len(commenti_non_3))

89


mi torna che ci siano 89 commenti che non sono stati presentati almeno 3 volte: 89 (minore 3) + 890 (maggiore di 3) = 979 (totale)

??? con gli 89 commenti che sono stati valutati <3 volto, posso pensare di fare un secondo round di valutazione così di ottinere un numero di risposte sufficienti anche per questi commenti ed utilizzarli all'interno dei vari modelli di regressione che andrò a creare.


In [39]:
# salvo il dataframe dopo aver eseguito un primo preprocessing
df.to_csv('responses_27mag_PREPROCESS.csv', index=False, encoding='utf-8', sep=';')